# 04 — Banc d'essai d'inférence et test d'intégration**CPU runtime, deliberately.** The production node is 1 vCPU / 1 GB. Benchmarking on a T4would produce numbers that describe hardware you do not deploy on.This notebook produces the figures for the slide currently titled *"Benchmark d'InférencePrévu"* — so it can stop saying *prévu*.Three measurements:1. **Local latency decomposition** — where the milliseconds actually go.2. **Mojo versus NumPy postprocessing** — the honest speedup, if the kernel builds.3. **End-to-end against the deployed endpoint** — what a user experiences.

## 1 — Setup

In [ ]:
!pip install -q onnxruntime==1.20.1 opencv-python-headless==4.10.0.84 \                requests pandas matplotlib ultralytics==8.3.55import os, subprocessif os.path.exists("/content/Nutrivision"):    !rm -rf /content/Nutrivisionsubprocess.run(["git","clone","--depth","1",                "https://github.com/Zen-Daitsu/Nutrivision.git",                "/content/Nutrivision"], check=True)%cd /content/Nutrivisionimport platform, onnxruntime as ortprint("cpu       :", platform.processor())print("cores     :", os.cpu_count())print("ort       :", ort.__version__)print("providers :", ort.get_available_providers())

In [ ]:
from google.colab import drivedrive.mount('/content/drive')ARTIFACTS = "/content/drive/MyDrive/Nutrivision/artifacts"import shutil, osos.makedirs("backend/models", exist_ok=True)# Prefer the trained model; fall back to stock COCO so the notebook runs regardless.if os.path.exists(f"{ARTIFACTS}/yolov8n-seg.onnx"):    shutil.copy2(f"{ARTIFACTS}/yolov8n-seg.onnx", "backend/models/model.onnx")    print("using trained NutriVision weights")else:    from ultralytics import YOLO    m = YOLO("yolov8n-seg.pt")    p = m.export(format="onnx", imgsz=640, opset=17, simplify=True, dynamic=False)    shutil.move(str(p), "backend/models/model.onnx")    print("using stock COCO weights — latency is representative, accuracy is not")print("size:", round(os.path.getsize("backend/models/model.onnx")/1e6, 1), "MB")

## 2 — Latency decompositionConstrained to a single intra-op thread, matching `NV_ORT_INTRA_THREADS=1` on the LightsailSmall tier. Fifty iterations after five warm-up passes, reported as median and p95 —the mean is the wrong statistic for latency, which is right-skewed.

In [ ]:
import numpy as np, cv2, time, onnxruntime as ortimport sys; sys.path.insert(0, "backend")from app import postprocessfrom app.inference import letterboxopts = ort.SessionOptions()opts.intra_op_num_threads = 1opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALLsess = ort.InferenceSession("backend/models/model.onnx", opts,                            providers=["CPUExecutionProvider"])inp = sess.get_inputs()[0].namerng = np.random.default_rng(0)frame = (rng.random((1080, 1920, 3)) * 255).astype(np.uint8)   # typical phone capturedef stage_preprocess():    padded, r, pad = letterbox(frame, 640)    rgb = cv2.cvtColor(padded, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0    return np.ascontiguousarray(np.transpose(rgb, (2,0,1))[None]), r, padtensor, ratio, pad = stage_preprocess()def bench(fn, n=50, warmup=5):    for _ in range(warmup):        fn()    ts = []    for _ in range(n):        t0 = time.perf_counter(); fn(); ts.append((time.perf_counter()-t0)*1000)    a = np.array(ts)    return {"median_ms": round(float(np.median(a)),2),            "p95_ms": round(float(np.percentile(a,95)),2),            "min_ms": round(float(a.min()),2)}outputs = sess.run(None, {inp: tensor})results = {  "preprocess (letterbox + normalise)": bench(stage_preprocess),  "onnx forward pass":                  bench(lambda: sess.run(None, {inp: tensor})),  "postprocess (decode + NMS, NumPy)":  bench(lambda: postprocess.decode(                                             outputs[0], outputs[1], 0.30, 0.50, 30)),}import pandas as pdlat = pd.DataFrame(results).Tlat["share_%"] = (lat["median_ms"] / lat["median_ms"].sum() * 100).round(1)lat

In [ ]:
import matplotlib.pyplot as pltfig, ax = plt.subplots(figsize=(11, 3.2))left = 0colors = ["#F2C14E", "#7FD1B9", "#E0715F"]for (name, row), col in zip(lat.iterrows(), colors):    ax.barh([0], row["median_ms"], left=left, color=col,            label=f"{name} — {row['median_ms']} ms ({row['share_%']}%)")    left += row["median_ms"]ax.set_yticks([]); ax.set_xlabel("milliseconds (median, 1 thread)")ax.set_title(f"Where the {left:.0f} ms goes")ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.35), ncol=1, frameon=False)plt.tight_layout(); plt.show()pp = lat.loc["postprocess (decode + NMS, NumPy)", "share_%"]print(f"Postprocessing is {pp}% of local compute.")print(f"Even an infinitely fast Mojo kernel caps the end-to-end gain at {pp}%.")

## 3 — Mojo versus NumPy postprocessingRuns only if `libnvpost.so` was built and uploaded to Drive. If the kernel is still blocked onthe Mojo 1.0 migration, this cell reports that and the notebook continues — the absence isitself a finding worth stating.

In [ ]:
import os, shutilSO = f"{ARTIFACTS}/libnvpost.so"if not os.path.exists(SO):    print("Mojo kernel not available.")    print("Production reports source='numpy'. See README section 7 for why.")    mojo_result = Noneelse:    os.makedirs("backend/mojo/build", exist_ok=True)    shutil.copy2(SO, "backend/mojo/build/libnvpost.so")    from app import mojo_bridge    if mojo_bridge.load("backend/mojo/build/libnvpost.so"):        mojo_result = bench(lambda: mojo_bridge.decode_nms(outputs[0], 0.30, 0.50, 30))        numpy_ms = lat.loc["postprocess (decode + NMS, NumPy)", "median_ms"]        speedup = numpy_ms / mojo_result["median_ms"]        print(f"NumPy  : {numpy_ms:.2f} ms")        print(f"Mojo   : {mojo_result['median_ms']:.2f} ms")        print(f"Speedup: {speedup:.2f}x on postprocessing")        print(f"End-to-end improvement: "              f"{(numpy_ms - mojo_result['median_ms']) / lat['median_ms'].sum() * 100:.1f}%")    else:        print("library present but failed to load")        mojo_result = None

### A note for the reportSlide 12 currently cites *"jusqu'à 35000x plus rapides que Python"*. That figure is Modular'sscalar Mandelbrot microbenchmark against pure CPython. It does not describe this pipeline,because ONNX Runtime is already compiled C++ with vendor-tuned kernels, and Mojo here replacesonly the decode and NMS loop.The defensible claim is the measured one above: a speedup of *X* on postprocessing, which is*Y* % of end-to-end latency. Stating a smaller number you can reproduce is stronger thanciting a larger one you cannot.

## 4 — End-to-end against the deployed endpointWhat the user actually experiences: capture, upload over the network, inference, USDA lookup,response. Network transit typically dominates local compute on a mobile connection.

In [ ]:
API = "https://nutrivision-api.ee6hya9ch5810.ca-central-1.cs.amazonlightsail.com"import requests, jsontry:    h = requests.get(f"{API}/healthz", timeout=10)    print(json.dumps(h.json(), indent=2))    LIVE = h.okexcept Exception as e:    print("endpoint unreachable:", e)    print("If you deleted the Lightsail service to save credits, recreate it or skip section 4.")    LIVE = False

In [ ]:
import glob, io, timeimport numpy as np, pandas as pd, cv2# Use calibration photographs if present, otherwise the test split.pool = sorted(glob.glob("/content/drive/MyDrive/Nutrivision/calibration/*.jpg"))[:10] \       or sorted(glob.glob("my_dataset/test/images/*"))[:10]rows = []if LIVE and pool:    for path in pool:        img = cv2.imread(path)        scale = min(1.0, 1280 / max(img.shape[:2]))        img = cv2.resize(img, None, fx=scale, fy=scale)        ok, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 85])        t0 = time.perf_counter()        r = requests.post(f"{API}/api/v1/analyze",                          files={"file": ("plate.jpg", buf.tobytes(), "image/jpeg")},                          data={"reference_width_mm": "85.6"}, timeout=45)        rtt = (time.perf_counter() - t0) * 1000        if r.ok:            d = r.json()            rows.append({                "file": os.path.basename(path),                "payload_kb": round(len(buf) / 1024, 1),                "round_trip_ms": round(rtt, 1),                "server_inference_ms": d["inference_ms"],                "server_postprocess_ms": d["postprocess_ms"],                "network_and_overhead_ms": round(rtt - d["inference_ms"] - d["postprocess_ms"], 1),                "source": d["source"],                "items": len(d["items"]),                "kcal": d["totals"]["calories"],            })        else:            rows.append({"file": os.path.basename(path), "round_trip_ms": round(rtt,1),                         "source": f"HTTP {r.status_code}"})e2e = pd.DataFrame(rows)e2e

In [ ]:
if len(e2e) and "server_inference_ms" in e2e:    fig, ax = plt.subplots(figsize=(12, 5))    idx = range(len(e2e))    ax.bar(idx, e2e["server_inference_ms"], color="#7FD1B9", label="ONNX forward")    ax.bar(idx, e2e["server_postprocess_ms"], bottom=e2e["server_inference_ms"],           color="#E0715F", label="postprocess")    ax.bar(idx, e2e["network_and_overhead_ms"],           bottom=e2e["server_inference_ms"] + e2e["server_postprocess_ms"],           color="#F2C14E", label="network + USDA + overhead")    ax.set_xticks(list(idx)); ax.set_xticklabels(e2e["file"], rotation=60, ha="right", fontsize=7)    ax.set_ylabel("milliseconds"); ax.legend()    ax.set_title("End-to-end latency decomposition, Colab client to ca-central-1")    plt.tight_layout(); plt.show()    print("median round trip :", round(float(e2e['round_trip_ms'].median()), 1), "ms")    print("p95 round trip    :", round(float(e2e['round_trip_ms'].quantile(0.95)), 1), "ms")    print("median server-side:", round(float((e2e['server_inference_ms'] +                                              e2e['server_postprocess_ms']).median()), 1), "ms")

## 5 — Response contract checkThe frontend binds to specific field names. This asserts the deployed API still honours theschema in `backend/app/schemas.py` — the same contract `tests/test_contract.py` enforces in CI,verified here against the live service rather than the source.

In [ ]:
REQUIRED_TOP  = {"items", "totals", "inference_ms", "postprocess_ms", "source"}REQUIRED_ITEM = {"class_id","name","confidence","box_xyxy","mask_area_px",                 "mass_g","mass_confidence","macros","fdc_id"}REQUIRED_MACRO = {"protein","carbs","fat","calories"}if LIVE and pool:    img = cv2.imread(pool[0])    ok, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 85])    d = requests.post(f"{API}/api/v1/analyze",                      files={"file": ("p.jpg", buf.tobytes(), "image/jpeg")},                      timeout=45).json()    checks = [("top-level fields", REQUIRED_TOP <= set(d)),              ("totals fields",    REQUIRED_MACRO == set(d["totals"]))]    if d["items"]:        checks.append(("item fields",  REQUIRED_ITEM <= set(d["items"][0])))        checks.append(("macro fields", REQUIRED_MACRO == set(d["items"][0]["macros"])))        checks.append(("mass_confidence valid",                       d["items"][0]["mass_confidence"] in {"high","medium","low"}))        checks.append(("USDA resolved (non-zero macros)",                       d["items"][0]["macros"]["calories"] > 0))    else:        print("no detections in this frame; item-level checks skipped")    for label, passed in checks:        print(f"  {'PASS' if passed else 'FAIL'}  {label}")    print()    print(json.dumps(d, indent=2)[:1200])

## 6 — Export the benchmarkCopy the printed table straight into the presentation. Every number is reproducible byrerunning this notebook, which is the property that distinguishes a benchmark from a claim.

In [ ]:
import json, datetime, platform, osbench_report = {    "generated_utc": datetime.datetime.utcnow().isoformat() + "Z",    "client": {"cores": os.cpu_count(), "processor": platform.processor()},    "model": {"file": "yolov8n-seg", "input": 640,              "size_mb": round(os.path.getsize("backend/models/model.onnx")/1e6, 1)},    "local_latency_ms": lat.to_dict(orient="index"),    "mojo_postprocess_ms": mojo_result,    "endpoint": API if LIVE else None,    "end_to_end": e2e.to_dict(orient="records") if len(e2e) else [],}with open("benchmark_report.json", "w") as f:    json.dump(bench_report, f, indent=2)import shutilshutil.copy2("benchmark_report.json", ARTIFACTS)if len(e2e):    e2e.to_csv(f"{ARTIFACTS}/end_to_end_latency.csv", index=False)print(json.dumps(bench_report["local_latency_ms"], indent=2))print("\nsaved to", ARTIFACTS)

## Reading the result| Observation | What it means for the architecture ||---|---|| Forward pass dominates local compute | Mojo cannot fix this. A smaller model or a GPU node can. || Postprocessing is a single-digit percentage | The honest ceiling on Mojo's end-to-end contribution. || Network exceeds server compute | Reduce upload size before optimising inference. `MAX_EDGE` in `camera.js` is the lever. || p95 far above median | Lightsail Small is 0.5 vCPU and contends. Medium removes the tail. |The general principle worth stating in the defence: the bottleneck was measured, not assumed.Most of the effort that went into the Mojo kernel targeted under ten percent of the latency —knowing that is a stronger result than the optimisation would have been.